## Topic: Document Loaders in LangChain

### Agenda
- 1. Introduction of Document Loaders

- 2. Complete Taxonomy of Loaders

- 2. Why Do We Need Them?

- 4. Practical Example of Document Loaders 

- 5. Complete Summary


### 1. Introduction of Document Loaders

- Definition:
    - A Document Loader is a LangChain component that reads data from an external source (files, websites, databases, APIs) and converts it into a standardized list of Document objects that the rest of the LangChain pipeline can process.
    

In [ ]:
"""  - The Big Picture
┌─────────────────────────────────────────────────────────────┐
│              WHERE LOADERS FIT IN THE RAG PIPELINE          │
│                                                             │
│  External Data          LangChain Pipeline                  │
│  ─────────────          ──────────────────                  │
│                                                             │
│  📄 PDF Files    ─┐                                         │
│  🌐 Websites     ─┤                                         │
│  🗄️ Databases    ─┼──► [LOADER] ──► [SPLITTER] ──► [EMBED]   │
│  📊 CSV/Excel    ─┤         │            │            │     │
│  🔌 APIs         ─┤    List[Doc]    Chunks       Vectors    │
│  📝 Notion/Drive ─┘                                         │
│                                                             │
│  The Loader is the BRIDGE between your raw data             │
│  and the LangChain ecosystem.                               │
└─────────────────────────────────────────────────────────────┘ 
"""

# Example:
""" 
Think of a Document Loader as a "universal translator" at the UN:

    🇫🇷 French delegate speaks French
    🇯🇵 Japanese delegate speaks Japanese
    🇧🇷 Brazilian delegate speaks Portuguese

The translator converts ALL languages into ONE common language
so everyone can understand each other.

Similarly, a Loader converts PDFs, HTML, SQL, CSV, etc.
into ONE common format: LangChain Document objects.


# The Document is LangChain's universal data container. Every loader outputs a list of these objects, regardless of the source format.
"""

### 2. Why Do We Need Them?

- 1. Problem 1: LLMs Can't Read Files Directly
    -   You can't just hand a PDF to an LLM
        - response = llm.invoke("Read this file: report.pdf")
    
    -  The LLM has NO access to your filesystem!

- 2. Problem 2: Data Comes in Many Formats

In [ ]:
"""  - Example : Problem 2: Data Comes in Many Formats
Your company data lives in:
  📄 500 PDF reports
  🌐 200 web pages of documentation
  🗄️ A PostgreSQL database with 1M rows
  📊 50 Excel spreadsheets
  📝 1000 Notion pages
  📧 10,000 emails

Each format needs a DIFFERENT parsing strategy.
Document Loaders handle this complexity for you.
"""

- 3. Problem 3: Standardization

In [ ]:
""" 
# Without Loaders — every format needs custom code
    pdf_text = extract_pdf("report.pdf")        # Custom PDF parser
    html_text = scrape_website("docs.com")      # Custom scraper
    csv_data = parse_csv("data.csv")            # Custom CSV reader
    #  Now you have 3 different data structures!

# With Loaders — everything becomes the same Document format
    pdf_docs = PyPDFLoader("report.pdf").load()
    web_docs = WebBaseLoader("docs.com").load()
    csv_docs = CSVLoader("data.csv").load()
    #  All return List[Document] — same structure!

"""

#### WHERE WE USE THESE
- To Build RAG Base Application: 
    - RAG is a technique that combines information retrieval with language generation,
    where a model retrieves relevant documents from a knowledge base and then uses
    them as context to generate accurate and grounded responses.


- Benefits of using RAG
    - 1. Use of up-to-date information
    - 2. Better privacy
    - 3. No limit of document size


In [ ]:
"""   - How Loaders Work Internally
┌─────────────────────────────────────────────────────────────┐
│              LOADER INTERNAL FLOW                           │
│                                                             │
│  1. CONNECT TO SOURCE                                       │
│     Open file / HTTP request / DB connection / API call     │
│                                                             │
│  2. READ RAW DATA                                           │
│     Read bytes from PDF, HTML from URL, rows from DB        │
│                                                             │
│  3. PARSE & EXTRACT TEXT                                    │
│     PDF → Extract text from each page                       │
│     HTML → Strip tags, extract body text                    │
│     CSV → Convert rows to text strings                      │
│     DB → Format query results as text                       │
│                                                             │
│  4. BUILD DOCUMENT OBJECTS                                  │
│     For each chunk of text:                                 │
│       Document(                                             │
│           page_content="extracted text",                    │
│           metadata={"source": "...", "page": N}             │
│       )                                                     │
│                                                             │
│  5. RETURN List[Document]                                   │
│     [Document(...), Document(...), Document(...)]           │
└─────────────────────────────────────────────────────────────┘


"""

In [ ]:
"""  - Two Key Fields:
┌─────────────────────────────────────────────────────────────┐
│                    Document OBJECT                          │
│                                                             │
│  ┌─────────────────────────────────────────────────────┐    │
│  │  page_content: str                                  │    │
│  │  ─────────────────                                  │    │
│  │  The actual text extracted from the source.         │    │
│  │  This is what gets embedded and sent to the LLM.    │    │
│  │                                                     │    │
│  │  Example: "Revenue grew 15% YoY to $2.4B..."        │    │
│  └─────────────────────────────────────────────────────┘    │
│                                                             │
│  ┌─────────────────────────────────────────────────────┐    │
│  │  metadata: dict                                     │    │
│  │  ─────────────                                      │    │
│  │  Key-value pairs about the document's origin.       │    │
│  │  Used for filtering, citation, and debugging.       │    │
│  │                                                     │    │
│  │  Example: {"source": "report.pdf", "page": 3}       │    │
│  └─────────────────────────────────────────────────────┘    │
└─────────────────────────────────────────────────────────────┘ 


"""

### 2. Complete Taxonomy of Loaders

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│                  DOCUMENT LOADER TAXONOMY                         │
│                                                                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │  📁 FILE-BASED LOADERS                                     │  │
│  │  ├── PyPDFLoader          → PDF files                      │  │
│  │  ├── PDFPlumberLoader     → PDF (better tables)            │  │
│  │  ├── PyPDFDirectoryLoader → Entire folder of PDFs          │  │
│  │  ├── TextLoader           → .txt files                     │  │
│  │  ├── CSVLoader            → .csv files                     │  │
│  │  ├── UnstructuredExcelLoader → Excel files                 │  │
│  │  ├── Docx2txtLoader       → Word documents                 │  │
│  │  ├── UnstructuredMarkdownLoader → Markdown files           │  │
│  │  ├── JSONLoader           → JSON files                     │  │
│  │  ├── BSHTMLLoader         → HTML files                     │  │
│  │  └── UnstructuredFileLoader → Auto-detect format           │  │
│  └────────────────────────────────────────────────────────────┘  │
│                                                                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │  🌐 WEB-BASED LOADERS                                      │  │
│  │  ├── WebBaseLoader        → Single/multiple URLs           │  │
│  │  ├── RecursiveUrlLoader   → Crawl entire website           │  │
│  │  ├── SitemapLoader        → Load from sitemap.xml          │  │
│  │  ├── PlaywrightURLLoader  → JavaScript-rendered pages      │  │
│  │  └── SeleniumURLLoader    → Dynamic web pages              │  │
│  └────────────────────────────────────────────────────────────┘  │
│                                                                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │  🗄️ DATABASE LOADERS                                       │  │
│  │  ├── SQLDatabaseLoader    → SQL query results              │  │
│  │  ├── PostgreSQLLoader     → PostgreSQL tables              │  │
│  │  ├── MongoDBLoader        → MongoDB collections            │  │
│  │  └── SnowflakeLoader      → Snowflake data warehouse       │  │
│  └────────────────────────────────────────────────────────────┘  │
│                                                                  │
│  ┌────────────────────────────────────────────────────────────┐  │
│  │  🔌 API & CLOUD LOADERS                                    │  │
│  │  ├── YouTubeLoader        → YouTube video transcripts      │  │
│  │  ├── WikipediaLoader      → Wikipedia articles             │  │
│  │  ├── ArxivLoader          → Academic papers                │  │
│  │  ├── NotionDBLoader       → Notion databases               │  │
│  │  ├── GoogleDriveLoader    → Google Drive files             │  │
│  │  ├── SlackDirectoryLoader → Slack exports                  │  │
│  │  ├── ConfluenceLoader     → Confluence wiki pages          │  │
│  │  ├── GitHubIssuesLoader   → GitHub issues & PRs            │  │
│  │  ├── HuggingFaceLoader    → HuggingFace datasets           │  │
│  │  └── S3FileLoader         → AWS S3 buckets                 │  │
│  └────────────────────────────────────────────────────────────┘  │
└──────────────────────────────────────────────────────────────────┘

"""

### 3. Practical Example of Document Loaders 

In [1]:
from langchain_community.document_loaders import TextLoader


# ---------------------------------------------------------
# Step 1: Create the loader
# ---------------------------------------------------------

loader = TextLoader("cricket.txt") # path of text file 

# loader = TextLoader("cricket.txt", encoding="utf-8")


# ---------------------------------------------------------
# Step 2: Load the document
# ---------------------------------------------------------

documents = loader.load()


# ---------------------------------------------------------
# Step 3: Check how many documents were loaded
# ---------------------------------------------------------

print(f"Number of documents: {len(documents)}")


# ---------------------------------------------------------
# Step 4: Display the document content
# ---------------------------------------------------------

print("\nDocument Content:")
print(documents[0].page_content)


# ---------------------------------------------------------
# Step 5: Display metadata
# ---------------------------------------------------------

print("\nMetadata:")
print(documents[0].metadata)

Number of documents: 1

Document Content:
Beneath the sun or floodlight's gleam,

Cricket lives like a waking dream.

A field of green, a willowed sound,

Where legends rise and tales are found.

From dusty lanes where barefoot boys,

Chase every run with shrieks of joy,

To packed arenas roaring loud,

The game unites a global crowd.

A coin is tossed, the captains stare,

As tension thickens in the air.

Bat or bowl? A choice so bold,

A story new begins, retold.

The openers walk, calm yet brave,

Each stride a wave upon the wave.

They face the ball with narrowed eyes,

As silence grips the watching skies.

The bowler runs, a rhythmic beat,

Like thunder galloping on feet.

A leather flash, a wooden crack—

The ball takes flight, then tumbles back.

A flick through square, a drive through mid,

A lofted shot the fielder missed.

A single, double, sprint for three,

The crowd erupts in ecstasy.

But not for long—the trap is set,

The spinner loops, the pitch is wet.

A sudden turn, 

### 4. Complete Summary

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│                    DOCUMENT LOADERS                              │
│                                                                  │
│  WHAT:  Read external data and convert to List[Document]         │
│  WHY:   LLMs can't access files/DBs/websites directly            │
│  WHERE: First step in the RAG pipeline                           │
│                                                                  │
│  OUTPUT:                                                         │
│    Document(                                                     │
│        page_content="extracted text",                            │
│        metadata={"source": "...", "page": N}                     │
│    )                                                             │
│                                                                  │
│  CATEGORIES:                                                     │
│    📁 File:     PDF, TXT, CSV, JSON, DOCX, HTML, Excel           │
│    🌐 Web:      URL, Recursive Crawl, Sitemap, JavaScript        │
│    🗄️ Database: SQL, PostgreSQL, MongoDB, Snowflake              │
│    🔌 API:      YouTube, Wikipedia, ArXiv, Notion, Drive         │
│                                                                  │
│  METHODS:                                                        │
│    .load()      → Load all at once (eager)                       │
│    .lazy_load() → Load one at a time (memory efficient)          │
│                                                                  │
│  PIPELINE:                                                       │
│    [Loader] → [Splitter] → [Embeddings] → [Vector Store]         │
│        ↑                                                         │
│    This is where it all begins!                                  │
│                                                                  │
│  GOLDEN RULE:                                                    │
│  "Load from ALL your data sources. The more context your RAG     │
│   system has, the better its answers will be."                   │
└──────────────────────────────────────────────────────────────────┘
"""